# 01 – Duomenų paruošimas ir atskaitos metodas

## Darbo tikslas

Šiame darbe sprendžiamas **vieno žingsnio į priekį laiko eilutės prognozavimo uždavinys**: pagal informaciją, turimą laiko momentu \(t\), prognozuojamas **bendras dviračių nuomų skaičius kitą valandą \(t+1\)**.

Modelio išvestis:

$$
\hat{y}_{t+1}
$$

čia \(\hat{y}_{t+1}\) – prognozuojama kitos valandos dviračių paklausa.

**Prognozės horizontas – 1 valanda.**

Praktinis tikslas – gauti prognozę, kuri galėtų padėti operatoriui iš anksto įvertinti artimiausios valandos dviračių poreikį ir planuoti jų perskirstymą. Šiame darbe pats perskirstymo optimizavimo uždavinys nesprendžiamas – vertinama tik paklausos prognozė.

### Šio Notebook paskirtis

1. nuskaityti ir chronologiškai paruošti duomenis;
2. suformuoti prognozavimo tikslą \(t+1\);
3. sudaryti tik prognozės momentu prieinamus požymius;
4. atskirti DEV ir galutinį TEST laikotarpius;
5. patikrinti pagrindines duomenų nutekėjimo (*data leakage*) rizikas;
6. įgyvendinti sezoninį **atskaitos metodą (baseline)**;
7. įvertinti jį tomis pačiomis metrikomis, kurios vėliau bus naudojamos intelektualiesiems modeliams.

> **Svarbu:** `casual` ir `registered` nenaudojami kaip modelių požymiai, nes pradiniame UCI rinkinyje bendras nuomų skaičius yra `cnt = casual + registered`. Galutinis TEST šiame etape nenaudojamas nei modeliui, nei hiperparametrams parinkti.

In [ ]:
from pathlib import Path
import sys, warnings, json
warnings.filterwarnings("ignore")
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA_PATH = ROOT / "hour.csv"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
from src.common import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Duomenų nuskaitymas ir prognozavimo uždavinio sudarymas

Naudojamas UCI **Bike Sharing** valandinis duomenų rinkinys `hour.csv`. Pradiniame rinkinyje `cnt` yra bendras per valandą išnuomotų dviračių skaičius.

Funkcija `load_prepare()`:

- suformuoja tikslų datos ir valandos `datetime`;
- surikiuoja duomenis chronologiškai;
- kitos valandos `cnt` reikšmę perkelia į tikslinį kintamąjį `target`;
- suformuoja `target_time = datetime + 1 val.`;
- sudaro vėlavimo ir slenkančios statistikos požymius;
- pašalina `casual`, `registered`, `instant`, `dteday` ir pradinį `cnt` iš modelių įvesties.

Taigi modelio uždavinys yra:

$$
X_t \longrightarrow y_{t+1}
$$

kur \(X_t\) – laiko momentu \(t\) prieinama informacija, o \(y_{t+1}\) – faktinė dviračių paklausa kitą valandą.

In [ ]:
work = load_prepare(DATA_PATH)
FEATURES = feature_columns(work)
dev, test = split_dev_test(work)

print("Naudojami požymiai:")
print(FEATURES)

print("\nDEV laikotarpis:")
print(dev["target_time"].min(), "→", dev["target_time"].max())
print("DEV stebėjimų skaičius:", len(dev))

print("\nTEST laikotarpis:")
print(test["target_time"].min(), "→", test["target_time"].max())
print("TEST stebėjimų skaičius:", len(test))

display(work.head())

### DEV ir TEST skaidymo logika

Duomenys **nemaišomi atsitiktinai**, nes tai yra laiko eilutės prognozavimo uždavinys. Ankstesnė duomenų dalis naudojama DEV etapui, o paskutinės 30 kalendorinių dienų intervalas rezervuojamas galutiniam TEST vertinimui.

`TEST_HOURS = 30 × 24 = 720` nusako **laiko intervalo ilgį**, tačiau UCI valandiniame rinkinyje ne kiekvienai kalendorinei valandai yra atskiras įrašas. Todėl po tikslaus \(t+1\) tikslo ir vėlavimo požymių sudarymo TEST eilučių skaičius gali būti mažesnis už 720. Svarbiausia, kad TEST laikotarpis yra chronologiškai vėlesnis už DEV ir nėra naudojamas hiperparametrams parinkti.

Toliau DEV dalyje naudojamas **slenkančios kilmės (rolling-origin)** vertinimas: mokymo istorija plečiama, o validavimas atliekamas vėlesniuose chronologiniuose segmentuose.

## 2. Pradinė duomenų analizė

Prieš mokant modelius svarbu pamatyti, kaip paklausa kinta laike ir paros metu. Toliau pateikiami du aprašomieji grafikai:

1. vienos savaitės paklausos kitimas;
2. vidutinė paklausa pagal paros valandą.

Šie grafikai **nėra modelio kokybės įrodymas**. Jie naudojami laiko eilutės struktūrai suprasti ir vėlesnių laiko bei sezoninių požymių naudojimui pagrįsti.

In [ ]:
# Vienos savaitės dviračių paklausos kitimas
plt.figure(figsize=(13, 4))

n = min(24 * 7, len(work))
plt.plot(
    work["datetime"].iloc[:n],
    work["target"].iloc[:n]
)

plt.title("Dviračių paklausos kitimas per vieną savaitę")
plt.xlabel("Laikas")
plt.ylabel("Nuomų skaičius")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


# Vidutinė dviračių paklausa pagal paros valandą
hourly = work.groupby("hr")["target"].mean()

plt.figure(figsize=(10, 4))
plt.bar(hourly.index, hourly.values)

plt.title("Vidutinė dviračių paklausa pagal paros valandą")
plt.xlabel("Valanda")
plt.ylabel("Vidutinis nuomų skaičius")
plt.xticks(range(24))
plt.tight_layout()
plt.show()

### Pradinės analizės interpretacija

Pirmos savaitės grafikas leidžia matyti, kad paklausa per parą stipriai kinta ir turi pasikartojančių pakilimų bei sumažėjimų.

Vidutinės paklausos pagal valandą grafikas leidžia įvertinti paros struktūrą: nakties valandomis paklausa yra mažesnė, o tam tikromis dienos valandomis matomi ryškesni paklausos padidėjimai.

Tai pagrindžia valandos, savaitės sezoniškumo ir ankstesnės paklausos požymių įtraukimą. Tačiau ar konkretūs požymiai iš tikrųjų pagerina prognozę, turi parodyti vėlesni eksperimentai, o ne vien ši aprašomoji analizė.

## 3. Modeliams naudojami požymiai

| Požymių grupė | Požymiai | Paskirtis |
|---|---|---|
| Kalendoriniai | `season`, `yr`, `mnth`, `hr`, `holiday`, `weekday`, `workingday` | Apibūdina laiko ir darbo / poilsio dienos kontekstą |
| Meteorologiniai | `weathersit`, `temp`, `atemp`, `hum`, `windspeed` | Apibūdina laiko momentu \(t\) turimas oro sąlygas |
| Vėlavimo | `lag_1`, `lag_24`, `lag_168` | Paklausa prieš 1, 24 ir 168 valandas |
| Slenkančios statistikos | `rolling_mean_24`, `rolling_mean_168` | Ankstesnės paros ir savaitės paklausos vidurkio informacija |

### Vėlavimo požymių reikšmė

- `lag_1` – paklausa prieš 1 valandą;
- `lag_24` – paklausa prieš 24 valandas;
- `lag_168` – paklausa prieš 168 valandas, t. y. prieš savaitę tuo pačiu laiku;
- `rolling_mean_24` – ankstesnių 24 valandų slenkantis vidurkis;
- `rolling_mean_168` – ankstesnių 168 valandų slenkantis vidurkis.

Slenkančios statistikos skaičiavime naudojamas poslinkis, todėl dabartinė ir būsima tikslinė reikšmė į vidurkį nepatenka.

> **Meteorologinių požymių apribojimas.** Šiame sprendime naudojamos laiko momento \(t\) faktinės oro sąlygos. Prognozuojamos valandos \(t+1\) faktiniai orai nenaudojami. Praktinėje sistemoje, jei būtų norima naudoti \(t+1\) orus, juos reikėtų pakeisti realiai tuo metu prieinama orų prognoze.

## 4. Duomenų nutekėjimo (*data leakage*) kontrolė

Laiko eilutės prognozėje modelis turi naudoti tik informaciją, kuri būtų prieinama **prognozės sudarymo momentu**. Jei į mokymą arba prognozę patektų būsima informacija, vertinimo metrikos galėtų atrodyti nepagrįstai geros.

Šiame darbe tikrinamos pagrindinės rizikos:

1. **`casual` ir `registered` nenaudojami kaip požymiai.** UCI rinkinyje:

$$
cnt = casual + registered
$$

Todėl šių dviejų stulpelių naudojimas prognozuojant `cnt` būtų tiesioginis tikslinio kintamojo nutekėjimas.

2. **Prognozės laikas yra vėlesnis už požymių laiką:**

$$
t_{\mathrm{target}} > t_{\mathrm{features}}
$$

3. **Vėlavimo požymiai sudaromi pagal tikslų laiką**, o ne vien pagal eilučių numerius. Tai svarbu, jei duomenyse nėra visų valandų.

4. **Slenkančios statistikos skaičiuojamos tik iš praeities.**

5. **DEV baigiasi prieš TEST**, todėl galutinis TEST nenaudojamas modelių ar hiperparametrų parinkimui.

Automatinės patikros yra papildomas įrodymas, tačiau jos nepakeičia rankinės požymių prieinamumo prognozės momentu peržiūros.

In [ ]:
def show_leakage_proof(work, dev, test, features):
    checks = {
        "Duomenys chronologine tvarka":
            bool(work["datetime"].is_monotonic_increasing),

        "Tikslas yra vėlesnis už požymių laiką":
            bool((work["target_time"] > work["datetime"]).all()),

        "casual nenaudojamas":
            "casual" not in features,

        "registered nenaudojamas":
            "registered" not in features,

        "DEV baigiasi prieš TEST":
            bool(dev["target_time"].max() < test["target_time"].min()),

        "TEST turi duomenų":
            len(test) > 0,
    }

    proof = pd.DataFrame({
        "Patikra": list(checks.keys()),
        "Rezultatas": [
            "PRAEITA" if value else "NEPRAEITA"
            for value in checks.values()
        ]
    })

    display(proof)

    if not all(checks.values()):
        raise ValueError(
            "Bent viena esminė duomenų nutekėjimo patikra NEPRAEITA. "
            "Eksperimento tęsti negalima."
        )

    print("IŠVADA: pagrindinės automatinės duomenų nutekėjimo patikros PRAEITOS.")
    print("Pastaba: tai nepakeičia rankinės požymių prieinamumo prognozės momentu peržiūros.")

    return proof


leakage_proof = show_leakage_proof(work, dev, test, FEATURES)

# Papildomi techniniai assert testai.
report = leakage_report(work, dev, test, FEATURES)

assert report["chronological_sorted"]
assert report["target_after_feature_time"]
assert not report["casual_in_features"]
assert not report["registered_in_features"]
assert report["dev_before_test"]

### Nutekėjimo patikros interpretacija

Jeigu visos lentelės eilutės pažymėtos **PRAEITA**, pagrindinės automatiškai tikrinamos sąlygos yra įvykdytos:

- duomenys išlaiko chronologinę tvarką;
- tikslinė valanda yra vėlesnė už požymių sudarymo laiką;
- `casual` ir `registered` nepatenka į modelio įvestį;
- DEV laikotarpis yra prieš galutinį TEST laikotarpį;
- TEST rinkinyje yra vertinimui skirtų duomenų.

Jeigu bent viena esminė patikra nepraeina, eksperimentas sustabdomas ir problema turi būti ištaisyta prieš mokant modelius.

## 5. Sezoninis atskaitos metodas (baseline)

Atskaitos metodas reikalingas tam, kad būtų galima patikrinti, ar sudėtingesni intelektualieji modeliai iš tikrųjų pagerina prognozę.

Šiame darbe naudojamas vienas sezoninis atskaitos metodas: kitos valandos paklausa prognozuojama pagal tos pačios savaitės dienos ir valandos paklausą per \(K\) ankstesnių savaičių.

### Atskaitos metodo formulė

$$
\hat{y}_{t+1}^{\mathrm{baseline}}
=
\frac{1}{K}
\sum_{k=1}^{K}
y_{t+1-168k}
$$

čia:

- \(\hat{y}_{t+1}^{\mathrm{baseline}}\) – prognozuojama kitos valandos dviračių paklausa;
- \(K\) – naudojamų ankstesnių savaičių skaičius;
- \(168 = 24 \times 7\) – valandų skaičius savaitėje;
- \(y_{t+1-168k}\) – tos pačios savaitės dienos ir valandos faktinė paklausa prieš \(k\) savaičių.

Šiame darbe naudojama:

$$
K = 4
$$

todėl prognozė gaunama iš iki keturių ankstesnių savaičių atitinkamos valandos paklausos vidurkio.

### Formulės ryšys su kodu

Formulę realizuoja funkcija:

```python
seasonal_baseline(train, valid, k_weeks=4)
```

Ryšys su formule:

- `k_weeks=4` atitinka \(K=4\);
- kiekvienai prognozuojamai valandai ieškoma \(t-168\), \(t-336\), \(t-504\) ir \(t-672\) valandų istorija;
- rastos ankstesnių savaičių reikšmės suvidurkinamos;
- jeigu konkrečiai valandai ankstesnės savaitinės reikšmės nerandamos, funkcijoje naudojamas mokymo dalies `target` vidurkis.

Taip formulė tiesiogiai susiejama su programine realizacija.

## 6. Chronologinis DEV vertinimas

Atskaitos metodas vertinamas ne viename atsitiktiniame padalijime, o keliuose **slenkančios kilmės (rolling-origin)** DEV skaidiniuose.

Kiekviename skaidinyje:

1. mokymo dalis yra chronologiškai ankstesnė;
2. validavimo dalis yra vėlesnė;
3. piko riba \(P90\) apskaičiuojama tik iš to skaidinio mokymo duomenų;
4. galutinis TEST nenaudojamas.

Tokios pačios DEV skaidymo ir metrikų sąlygos bus taikomos intelektualiesiems modeliams.

In [ ]:
folds = rolling_origin_folds(dev)

rows = []

for i, (tri, vai) in enumerate(folds, 1):
    tr = dev.iloc[tri]
    va = dev.iloc[vai]

    pred = seasonal_baseline(
        tr,
        va,
        k_weeks=4
    )

    # Piko riba nustatoma tik pagal konkretaus skaidinio mokymo duomenis.
    peak_threshold = float(
        tr["target"].quantile(PEAK_Q)
    )

    rows.append({
        "fold": i,
        **metrics(
            va["target"],
            pred,
            peak_threshold
        )
    })

baseline_dev = pd.DataFrame(rows)

print("Atskaitos metodo rezultatai kiekviename DEV skaidinyje:")
display(baseline_dev)

# Bendram palyginimui vidurkinamos tik pagrindinės vertinimo metrikos.
baseline_summary = (
    baseline_dev[
        ["MAE", "RMSE", "Peak_MAE", "PUR"]
    ]
    .mean()
    .to_frame("DEV vidurkis")
)

print("Pagrindinių metrikų vidurkis per DEV skaidinius:")
display(baseline_summary)

print("\nREZULTATŲ SANTRAUKA")
print(f"Vidutinė MAE: {baseline_dev['MAE'].mean():.2f} nuomos.")
print(f"Vidutinė RMSE: {baseline_dev['RMSE'].mean():.2f} nuomos.")
print(f"Vidutinė Peak MAE: {baseline_dev['Peak_MAE'].mean():.2f} nuomos.")
print(f"Vidutinis PUR: {baseline_dev['PUR'].mean() * 100:.1f} %.")

### Rezultatų lentelių paaiškinimas

Pirmoje lentelėje pateikiami rezultatai atskiruose chronologiniuose DEV skaidiniuose.

- **`fold`** – DEV skaidinio numeris;
- **MAE** – vidutinė absoliuti klaida;
- **RMSE** – kvadratinės paklaidos šaknis;
- **Peak_MAE** – vidutinė absoliuti klaida tik piko valandomis;
- **PUR** – piko valandų dalis, kai prognozė buvo mažesnė už faktinę paklausą;
- **N** – vertintų stebėjimų skaičius;
- **N_peak** – piko stebėjimų skaičius.

Antroje lentelėje pateikiamas tik keturių pagrindinių metrikų vidurkis. `fold`, `N` ir `N_peak` nėra vidurkinami bendram modelių palyginimui, nes jie nėra modelio kokybės metrikos.

**MAE, RMSE, Peak_MAE ir PUR atveju mažesnė reikšmė reiškia geresnį rezultatą.**

In [ ]:
# Atskaitos metodo MAE ir RMSE stabilumas skirtinguose DEV skaidiniuose
x = np.arange(len(baseline_dev))
width = 0.35

plt.figure(figsize=(9, 4))

plt.bar(
    x - width / 2,
    baseline_dev["MAE"],
    width=width,
    label="MAE"
)

plt.bar(
    x + width / 2,
    baseline_dev["RMSE"],
    width=width,
    label="RMSE"
)

plt.xticks(
    x,
    [f"DEV {int(v)}" for v in baseline_dev["fold"]]
)

plt.title("Atskaitos metodo klaidos skirtinguose DEV skaidiniuose")
plt.xlabel("Chronologinis DEV skaidinys")
plt.ylabel("Klaida, nuomų skaičius")
plt.legend()
plt.tight_layout()
plt.show()


# Automatinė stabilumo santrauka, kad tekstas visada atitiktų gautus rezultatus.
best_fold = baseline_dev.loc[baseline_dev["MAE"].idxmin()]
worst_fold = baseline_dev.loc[baseline_dev["MAE"].idxmax()]

print(
    f"Mažiausia MAE gauta DEV {int(best_fold['fold'])}: "
    f"{best_fold['MAE']:.2f}."
)
print(
    f"Didžiausia MAE gauta DEV {int(worst_fold['fold'])}: "
    f"{worst_fold['MAE']:.2f}."
)
print(
    f"MAE skirtumas tarp šių skaidinių: "
    f"{worst_fold['MAE'] - best_fold['MAE']:.2f} nuomos."
)

### Rezultatų stabilumo vertinimas

Jeigu MAE ir RMSE reikšmės tarp DEV skaidinių skiriasi, tai rodo, kad metodo tikslumas priklauso nuo vertinamo laikotarpio. Dėl šios priežasties vieno atsitiktinio duomenų padalijimo nepakanka.

Keli chronologiniai DEV skaidiniai leidžia įvertinti, ar modelio rezultatas yra pakankamai stabilus skirtingais vėlesniais laiko intervalais. Tas pats vertinimo principas naudojamas ir intelektualiesiems modeliams.

## 7. Faktinės ir prognozuotos paklausos palyginimas

Vien metrikų nepakanka. Toliau paskutiniame DEV skaidinyje vizualiai palyginama faktinė paklausa ir sezoninio atskaitos metodo prognozė.

Tai leidžia patikrinti, ar metodas seka bendrą paklausos kitimą ir kur atsiranda didesni neatitikimai.

In [ ]:
tri, vai = folds[-1]
tr = dev.iloc[tri]
va = dev.iloc[vai]

baseline_pred = seasonal_baseline(
    tr,
    va,
    k_weeks=4
)

# Rodoma pirmoji paskutinio DEV skaidinio savaitė.
n = min(168, len(va))

plt.figure(figsize=(13, 4))

plt.plot(
    va["target_time"].iloc[:n],
    va["target"].iloc[:n],
    label="Faktinė paklausa",
    linewidth=2
)

plt.plot(
    va["target_time"].iloc[:n],
    baseline_pred[:n],
    label="Atskaitos metodo prognozė"
)

plt.title("Atskaitos metodas: faktinė ir prognozuota paklausa")
plt.xlabel("Laikas")
plt.ylabel("Nuomų skaičius")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

last_peak = float(
    tr["target"].quantile(PEAK_Q)
)

last_metrics = metrics(
    va["target"],
    baseline_pred,
    last_peak
)

print("Paskutinio DEV skaidinio metrikos:")
display(
    pd.Series(
        last_metrics,
        name="Reikšmė"
    )
)

### Prognozės grafiko interpretavimo principas

Grafike vertinama ne tik tai, ar prognozė atkartoja bendrą paklausos kryptį, bet ir ar tiksliai atkuriami didesnės paklausos laikotarpiai.

Jeigu faktinės ir prognozuotos kreivės piko metu ryškiai išsiskiria, tai atsispindi **Peak MAE** ir **PUR** metrikose. Tai svarbu praktiniu požiūriu, nes per maža prognozė didelės paklausos metu gali reikšti nepakankamą pasiruošimą realiam dviračių poreikiui.

Galutinė išvada, ar intelektualieji modeliai pagerina šį rezultatą, bus daroma tik juos palyginus tomis pačiomis sąlygomis.

## 8. Vertinimo metrikos ir jų formulės

Visiems modeliams naudojamos tos pačios vertinimo metrikos.

### 8.1. Vidutinė absoliuti klaida (MAE)

$$
\mathrm{MAE}
=
\frac{1}{N}
\sum_{i=1}^{N}
\left|y_i-\hat{y}_i\right|
$$

čia:

- \(y_i\) – faktinė paklausa;
- \(\hat{y}_i\) – prognozuota paklausa;
- \(N\) – vertinamų stebėjimų skaičius.

MAE parodo, kiek nuomų prognozė **vidutiniškai** skiriasi nuo faktinės paklausos.

### 8.2. Kvadratinės paklaidos šaknis (RMSE)

$$
\mathrm{RMSE}
=
\sqrt{
\frac{1}{N}
\sum_{i=1}^{N}
\left(y_i-\hat{y}_i\right)^2
}
$$

RMSE stipriau reaguoja į dideles prognozavimo klaidas, todėl papildo MAE.

### 8.3. Piko absoliuti klaida (Peak MAE)

Pirmiausia iš konkretaus mokymo rinkinio nustatoma piko riba:

$$
q_{0.90}
=
Q_{0.90}\left(y_{\mathrm{train}}\right)
$$

Tada piko stebėjimų aibė:

$$
P
=
\left\{
i : y_i > q_{0.90}
\right\}
$$

ir:

$$
\mathrm{Peak\ MAE}
=
\frac{1}{|P|}
\sum_{i\in P}
\left|y_i-\hat{y}_i\right|
$$

Piko riba nustatoma **tik iš mokymo duomenų**, o ne iš galutinio TEST.

### 8.4. Piko neįvertinimo dažnis (PUR)

$$
\mathrm{PUR}
=
\frac{
\sum_{i\in P}
\mathbb{1}
\left(\hat{y}_i < y_i\right)
}{
|P|
}
$$

čia \(\mathbb{1}(\cdot)\) – indikatorinė funkcija, lygi 1, kai sąlyga tenkinama.

PUR parodo, kokią faktinių piko valandų dalį modelis **prognozavo per mažai**.

> **Visų keturių metrikų atveju mažesnė reikšmė reiškia geresnį rezultatą.** PUR visada interpretuojama kartu su MAE, RMSE ir Peak MAE, nes vien tik piko neįvertinimo mažinimas galėtų skatinti sistemingą paklausos pervertinimą.

## 9. 01 etapo išvada

Šiame Notebook paruošta bendra duomenų ir vertinimo grandinė, kuri bus naudojama tolesniems modeliams:

1. suformuotas **kitos valandos** paklausos prognozavimo uždavinys;
2. duomenys išlaikyti chronologine tvarka;
3. sudaryti kalendoriniai, meteorologiniai, vėlavimo ir slenkančios statistikos požymiai;
4. `casual` ir `registered` pašalinti iš įvesties dėl tiesioginio ryšio su `cnt`;
5. DEV ir galutinis TEST atskirti chronologiškai;
6. įgyvendintos automatinės pagrindinių duomenų nutekėjimo sąlygų patikros;
7. įgyvendintas sezoninis atskaitos metodas ir parodytas jo formulės ryšys su kodu;
8. atskaitos metodas įvertintas keliuose chronologiniuose DEV skaidiniuose;
9. apibrėžtos bendros MAE, RMSE, Peak MAE ir PUR metrikos.

Toliau intelektualieji modeliai bus vertinami **ta pačia duomenų paruošimo logika, tais pačiais DEV skaidiniais ir tomis pačiomis metrikomis**.

Galutinis TEST rinkinys turi likti nepanaudotas hiperparametrų parinkimo metu. Jis bus naudojamas tik galutiniam jau užfiksuotų modelių palyginimui.

### Kas sąmoningai paliekama kitiems Notebook

Šiame 01 faile neatliekami:

- MLP, CatBoost, ansamblio ar N-BEATS mokymai;
- hiperparametrų paieška;
- galutinis TEST modelių palyginimas;
- abliacijos ir atsparumo bandymai;
- galutinė modelių klaidų analizė.

Šios dalys atliekamos vėlesniuose Notebook, kad duomenų paruošimas, modelių eksperimentai ir galutinis vertinimas būtų aiškiai atskirti.